# Notebook 01: Raw Dataset Inspection, Preprocessing & Tokenization

**Project:** RNN-based Sentiment Analysis using NLP  
**Dataset:** Stanford IMDB Large Movie Review Dataset (50,000 raw text reviews)  
**Objectives:**
1. Inspect raw text reviews and baseline statistics.
2. Perform text cleaning, lowercasing, HTML stripping, and deduplication.
3. Fit Keras Tokenizer on training set and inspect vocabulary rankings.
4. Pad sequences to fixed uniform length for Recurrent Neural Network input.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add project root to sys.path
sys.path.insert(0, "..")
from src.config import MAX_VOCAB_SIZE, MAX_SEQUENCE_LENGTH
from src.preprocessor import clean_text, fit_tokenizer, transform_texts
from src.data_loader import load_raw_data, preprocess_dataset, split_data

DATA_PATH = Path("../data/raw/imdb_reviews.csv")
print(f"Dataset exists: {DATA_PATH.exists()}")

## 1. Load Raw Dataset & Inspect First 5 Rows

In [ ]:
df = pd.read_csv(DATA_PATH)
pd.set_option('display.max_colwidth', 150)
df.head()

## 2. Dataset Shape, Missing Values, and Duplicates

In [ ]:
print("Shape:", df.shape)
print("Missing values:\n", df.isnull().sum())
print("Duplicates count:", df.duplicated(subset=['review']).sum())
print("\nClass balance:")
print(df['sentiment'].value_counts())

## 3. Apply Text Cleaning and Deduplication

In [ ]:
df_clean = preprocess_dataset(df)
print("Cleaned DataFrame shape:", df_clean.shape)
df_clean[['clean_review', 'label']].head()

## 4. Stratified Split (Train: 80%, Validation: 10%, Test: 10%)

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df_clean)
print(f"Train size: {len(X_train)} ({len(X_train)/len(df_clean)*100:.1f}%)")
print(f"Val size:   {len(X_val)} ({len(X_val)/len(df_clean)*100:.1f}%)")
print(f"Test size:  {len(X_test)} ({len(X_test)/len(df_clean)*100:.1f}%)")

## 5. Sequence Length Distribution Analysis
Determine the optimal `MAX_SEQUENCE_LENGTH` to capture review context while bounding RNN compute.

In [ ]:
train_lengths = X_train.apply(lambda text: len(text.split()))
print(f"Mean word length:   {train_lengths.mean():.1f}")
print(f"Median word length: {train_lengths.median():.1f}")
print(f"90th percentile:    {np.percentile(train_lengths, 90):.1f}")
print(f"95th percentile:    {np.percentile(train_lengths, 95):.1f}")

coverage_200 = (train_lengths <= MAX_SEQUENCE_LENGTH).mean() * 100
print(f"Percentage of reviews fully covered by MAX_LEN={MAX_SEQUENCE_LENGTH}: {coverage_200:.2f}%")

## 6. Fitting Tokenizer (Vocabulary Construction)

In [ ]:
tokenizer = fit_tokenizer(X_train.tolist(), max_words=MAX_VOCAB_SIZE)
print(f"Total unique vocabulary words found: {len(tokenizer.word_index)}")
print(f"Active vocabulary limit (num_words): {tokenizer.num_words}")

# Top 15 most frequent words and their assigned token IDs
top_15 = list(tokenizer.word_index.items())[:15]
print("\nTop 15 Most Frequent Tokens & IDs:")
for word, idx in top_15:
    print(f"  '{word}': {idx}")

## 7. Sequence Padding and Tensor Shapes

In [ ]:
X_train_pad = transform_texts(tokenizer, X_train.tolist(), max_len=MAX_SEQUENCE_LENGTH)
X_val_pad = transform_texts(tokenizer, X_val.tolist(), max_len=MAX_SEQUENCE_LENGTH)
X_test_pad = transform_texts(tokenizer, X_test.tolist(), max_len=MAX_SEQUENCE_LENGTH)

print("Final Tensor Dimensions:")
print(f"  X_train_pad: {X_train_pad.shape}  |  y_train: {y_train.shape}")
print(f"  X_val_pad:   {X_val_pad.shape}   |  y_val:   {y_val.shape}")
print(f"  X_test_pad:  {X_test_pad.shape}   |  y_test:  {y_test.shape}")